In [3]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import io, zipfile
import polars as pl
import config

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
path = sorted((config.RAW_DIR / "funding" / "ETHUSDT").glob("*.zip"))[0]   # первый месяц
print(path.name)

with zipfile.ZipFile(path) as z:
    raw = z.read(z.namelist()[0])     # CSV внутри zip

print(raw[:300].decode())             # первые строки как текст: видно заголовок и формат
df_f = pl.read_csv(io.BytesIO(raw))
print(df_f.shape)
df_f.head()

ETHUSDT-fundingRate-2024-10.zip
calc_time,funding_interval_hours,last_funding_rate
1727740800000,8,0.00010000
1727769600000,8,0.00010000
1727798400001,8,0.00008397
1727827200000,8,0.00010000
1727856000000,8,0.00006859
1727884800000,8,0.00010000
1727913600000,8,0.00002418
1727942400000,8,0.00004124
1727971200000,8,0.00004621
172800
(93, 3)


calc_time,funding_interval_hours,last_funding_rate
i64,i64,f64
1727740800000,8,0.0001
1727769600000,8,0.0001
1727798400001,8,0.000084
1727827200000,8,0.0001
1727856000000,8,0.000069


## Фандинг: проверка данных

Все месяцы ETH и BTC: число выплат, интервалы, ставки, совпадение со свечами 5m.

In [5]:
from src.funding import load_funding, funding_on_bars

pair = pl.read_parquet(config.PROCESSED_DIR / "pair_5m.parquet")

for sym in (config.SYMBOL_A, config.SYMBOL_B):
    f = load_funding(sym)
    on_bars = funding_on_bars(pair["open_time"].to_numpy(), f)
    gaps_h = f["funding_time"].diff().drop_nulls() / 3_600_000      # интервал между выплатами в часах
    print(f"--- {sym} ---")
    print("выплат:", f.height, "| попало на свечи:", int((on_bars != 0).sum()))
    print("интервалы, ч:", gaps_h.unique().sort().to_list())
    print("ставка: средняя", round(f["rate"].mean() * 1e4, 3), "bp | мин",
          round(f["rate"].min() * 1e4, 2), "bp | макс", round(f["rate"].max() * 1e4, 2), "bp")

--- ETHUSDT ---
выплат: 2190 | попало на свечи: 2190
интервалы, ч: [7.999999999999999]
ставка: средняя 0.452 bp | мин -3.65 bp | макс 5.76 bp
--- BTCUSDT ---
выплат: 2190 | попало на свечи: 2190
интервалы, ч: [7.999999999999999]
ставка: средняя 0.48 bp | мин -1.52 bp | макс 7.24 bp


## Бэктест с фандингом против v1

Те же правила; фандинг списывается в момент выплаты с позиции, открытой в этот момент.
Проверяем: (1) сделки те же, (2) на сколько изменился PnL.

In [6]:
import numpy as np
from src.signals import log_spread, rolling_zscore
from src.engine import run_backtest

df = pair                                                   # pair_5m уже загружен выше
z = rolling_zscore(log_spread(df["close_a"].to_numpy(), df["close_b"].to_numpy()), config.WINDOW)
tradable = ((df["n_traded_a"] > 0) & (df["n_traded_b"] > 0)).to_numpy()
open_time = df["open_time"].to_numpy()

fa = funding_on_bars(open_time, load_funding(config.SYMBOL_A))   # ставка ETH на каждой свече
fb = funding_on_bars(open_time, load_funding(config.SYMBOL_B))   # ставка BTC на каждой свече

args = (df["open_a"].to_numpy(), df["open_b"].to_numpy(),
        df["close_a"].to_numpy(), df["close_b"].to_numpy(), tradable, z)

In [7]:
rows = []
for tier in ("regular", "vip4", "vip7"):
    a = run_backtest(*args, tier)                 # v1 без фандинга
    b = run_backtest(*args, tier, fa, fb)         # с фандингом

    same = [(t["entry_t"], t["exit_t"], t["direction"]) for t in a["trades"]] == \
           [(t["entry_t"], t["exit_t"], t["direction"]) for t in b["trades"]]
    funding_total = sum(t["funding"] for t in b["trades"])

    rows.append({
        "tier": tier,
        "сделки те же": same,
        "капитал без фандинга": round(a["equity"][-1], 2),
        "капитал с фандингом": round(b["equity"][-1], 2),
        "разница": round(b["equity"][-1] - a["equity"][-1], 2),
        "фандинг по сделкам": round(funding_total, 2),
    })

pl.DataFrame(rows)

tier,сделки те же,капитал без фандинга,капитал с фандингом,разница,фандинг по сделкам
str,bool,f64,f64,f64,f64
"""regular""",true,1211.83,1394.26,182.43,182.43
"""vip4""",true,9110.21,9292.63,182.43,182.43
"""vip7""",true,12269.56,12451.99,182.43,182.43


In [8]:
b = run_backtest(*args, "regular", fa, fb)
f = np.array([t["funding"] for t in b["trades"]])
print("сделок с фандингом ≠ 0:", int((f != 0).sum()), "из", len(f))
print("средний фандинг на сделку: $", round(f.mean(), 2))
print("лучший / худший фандинг по сделке: $", round(f.max(), 2), "/", round(f.min(), 2))
print("фандинг в % от итогового PnL:", round(100 * f.sum() / (b["equity"][-1] - config.CAPITAL), 1), "%")

сделок с фандингом ≠ 0: 668 из 987
средний фандинг на сделку: $ 0.18
лучший / худший фандинг по сделке: $ 6.01 / -3.56
фандинг в % от итогового PnL: -1.0 %


### Выводы: фандинг

- Сделки с фандингом и без — те же (входы, выходы, направления) на всех уровнях → фандинг не влияет на решения.
- Фандинг за 2 года: +$182.43 (получали больше, чем платили), одинаково на всех уровнях;
  разница капитала = сумма фандинга по сделкам → учёт верный.
- Капитал с фандингом: Regular $1 394 / VIP-4 $9 293 / VIP-7 $12 452 (было $1 212 / $9 110 / $12 270).
- 668 из 987 сделок застали выплату; в среднем +$0.18 на сделку против комиссий $12.8–24.
- Фандинг уменьшает убыток примерно на 1% → в паре ETH/BTC он почти гасится; картину не меняет.